## Loading All 4 Feature Files

In [20]:
import pandas as pd
import numpy as np

df_billings = pd.read_csv('../../../data/interim/billings_features.csv')
df_renewal  = pd.read_csv('../../../data/interim/renewal_calls_features.csv')
df_email    = pd.read_csv('../../../data/interim/email_features.csv')
df_cc       = pd.read_csv('../../../data/interim/cc_calls_features.csv')

print(f"Billings : {df_billings.shape}")
print(f"Renewal Calls : {df_renewal.shape}")
print(f"Emails : {df_email.shape}")
print(f"CC Calls: {df_cc.shape}")

Billings : (113769, 64)
Renewal Calls : (14655, 38)
Emails : (21362, 32)
CC Calls: (14988, 45)



## Verifying co_ref is in All Files

In [21]:

for name, df in [('billings', df_billings), ('renewal', df_renewal),
                 ('email', df_email), ('cc', df_cc)]:
    has = 'co_ref' in df.columns
    unique = df['co_ref'].nunique() if has else 0
    print(f"{name:12s} — co_ref: {has} | unique customers: {unique:,}")

billings     — co_ref: True | unique customers: 46,328
renewal      — co_ref: True | unique customers: 14,655
email        — co_ref: True | unique customers: 21,362
cc           — co_ref: True | unique customers: 14,988


## Join All Datasets onto Billings

In [22]:
# Billings is the base — left join everything onto it
df = df_billings.copy()

df = df.merge(df_renewal, on='co_ref', how='left')
df = df.merge(df_email,   on='co_ref', how='left')
df = df.merge(df_cc,      on='co_ref', how='left')

print(f"Shape after join: {df.shape}")


Shape after join: (113769, 176)


In [23]:
print(df.columns.tolist())

['co_ref', 'renewal_month', 'sustainability_score', 'total_renewal_score_new', 'last_years_price', 'auto_renewal_score', 'status_scores', 'anchoring_score', 'tenure_scores', 'proforma_auto_renewal', 'proforma_world_pay_token', 'proforma_date', 'current_anchorings', 'payment_timeframe', 'registration_date', 'proforma_audit_status', 'renewal_score_at_release', 'proforma_approved_lists', 'tenure_years', 'prospect_renewal_date', 'starting_net', 'starting_vat', 'starting_gross', 'starting_membership_net', 'starting_package_net', 'starting_pqq_net', 'membership_net', 'package_net', 'pqqnet', 'prospect_outcome', 'total_amount', 'last_renewal', 'last_total_net_paid', 'last_connections', 'renewal_year', 'is_first_year', 'has_discount', 'discount_pct', 'price_increase_abs', 'price_increase_pct', 'price_increased_flag', 'band_ordinal', 'last_band_ordinal', 'band_changed', 'band_upgraded', 'band_downgraded', 'tenure_group_ordinal', 'anchor_group_ordinal', 'membership_status_ordinal', 'has_auto_ren

In [24]:
df.rename(columns={'high_risk_call_y': 'high_risk_call'}, inplace=True)
df.rename(columns={'dissatisfaction_issue_count_y': 'dissatisfaction_issue_count'}, inplace=True)


## Fill NaN from Unmatched Rows
Customers with no calls/emails in the window get NaN — fill with 0 since "no interaction" = 0.

In [25]:

renewal_cols = [c for c in df_renewal.columns if c != 'co_ref']
email_cols  = [c for c in df_email.columns if c != 'co_ref']
cc_cols   = [c for c in df_cc.columns if c != 'co_ref']
df[renewal_cols] = df[renewal_cols].fillna(0)
df[email_cols]  = df[email_cols].fillna(0)
df[cc_cols]    = df[cc_cols].fillna(0)



## Adding "Has Interaction" Flags

In [26]:
df['has_renewal_call'] = (df['rc_14d_total_calls'] > 0).astype(int)
df['has_email']   = (df['email_count'] > 0).astype(int)
df['has_cc_call'] = (df['cc_total_calls'] > 0).astype(int)

C:\Users\gopik\AppData\Local\Temp\ipykernel_14752\820585843.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['has_renewal_call'] = (df['rc_14d_total_calls'] > 0).astype(int)
C:\Users\gopik\AppData\Local\Temp\ipykernel_14752\820585843.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['has_email']   = (df['email_count'] > 0).astype(int)
C:\Users\gopik\AppData\Local\Temp\ipykernel_14752\820585843.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, wh

## Dropping co_ref and Remaining Non-Feature Columns


In [27]:
drop_cols = [
    'co_ref',           # ID — not a feature
    'renewal_month',    # raw date — not usable by model directly
]

df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
print(f"Shape after dropping ID cols: {df.shape}")

Shape after dropping ID cols: (113769, 177)


## Exporting final dataset

In [28]:
df.to_csv('../../../data/processed/final_dataset.csv', index=False)